In [3]:
import os
import hashlib
from pathlib import Path
from urllib.parse import urlparse

import pandas as pd
import requests

def ensure_https(url: str) -> str:
    if url is None or (isinstance(url, float) and pd.isna(url)):
        return None
    url = str(url).strip()
    if not url:
        return None
    if url.startswith("//"):
        return "https:" + url
    if not url.startswith(("http://", "https://")):
        return "https://" + url
    return url

def make_image_id_from_url(url: str) -> str:
    # Deterministic unique id (same url -> same id)
    return hashlib.sha1(url.encode("utf-8")).hexdigest()  # 40 hex chars

def download_one(url: str, out_path: Path, timeout=30) -> bool:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    try:
        with requests.get(url, stream=True, timeout=timeout) as r:
            r.raise_for_status()
            with open(out_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=1024 * 256):
                    if chunk:
                        f.write(chunk)
        return True
    except Exception:
        return False

def guess_ext_from_url(url: str, default=".jpg") -> str:
    try:
        p = urlparse(url).path
        ext = os.path.splitext(p)[1].lower()
        if ext in {".jpg", ".jpeg", ".png", ".webp", ".gif", ".tif", ".tiff"}:
            return ext
    except Exception:
        pass
    return default

def download_images_from_df(
    df: pd.DataFrame,
    url_col: str = "identifier",
    out_dir: str = "images",
) -> pd.DataFrame:
    df = df.copy()

    # normalize URLs
    df["image_url"] = df[url_col].apply(ensure_https)

    # make ids
    df["image_id"] = df["image_url"].apply(lambda u: None if u is None else make_image_id_from_url(u))

    # local paths
    out_dir = Path(out_dir)
    df["file_ext"] = df["image_url"].apply(lambda u: None if u is None else guess_ext_from_url(u))
    df["local_path"] = df.apply(
        lambda r: None if r["image_id"] is None else str(out_dir / f"{r['image_id']}{r['file_ext']}"),
        axis=1
    )

    # download (skips if already exists)
    ok_list = []
    for url, lp in zip(df["image_url"], df["local_path"]):
        if url is None or lp is None:
            ok_list.append(False)
            continue
        lp = Path(lp)
        if lp.exists() and lp.stat().st_size > 0:
            ok_list.append(True)
            continue
        ok_list.append(download_one(url, lp))
    return df


In [ ]:
from pathlib import Path
import pandas as pd

file_name = "output_file.parquet"
out_dir_name = "images"

df = pd.read_parquet(file_name)
out_dir = Path(out_dir_name)
out_dir.mkdir(parents=True, exist_ok=True)

df = download_images_from_df(df, url_col="identifier", out_dir=out_dir)

df.to_parquet(file_name)